[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/brenoprato/calculo-numerico/blob/main/atividades/atv_3/resolucao.ipynb)

# Atividade Prática III: Zeros de Funções Reais
## Métodos da Bisseção, Newton-Raphson e Secante
**Disciplina:** Cálculo Numérico  
**Docente:** Profª. Raquel J. Lobosco  
**Universidade Federal de São Paulo (UNIFESP)**

---
**Nome do integrante:** Breno Prato  
**Divisão de tarefas:** Implementação modular dos métodos em Python, estudo comparativo de convergência, confecção dos gráficos e geração das animações em formato GIF.

---

## 1. Enunciado e Objetivos
Em ciências e engenharia, determinar o valor de uma variável que anula uma função ($f(r) = 0$) é um problema clássico conhecido como cálculo de raízes ou zeros de funções reais. Na grande maioria das aplicações práticas, não existem fórmulas analíticas diretas, sendo imprescindível recorrer a **métodos numéricos iterativos**.

Os objetivos principais desta atividade são:
1. Implementar de forma modular, robusta e comentada os métodos da **Bisseção**, de **Newton-Raphson** e da **Secante** em Python;
2. Configurar critérios estritos de parada por tolerância ($10^{-6}$) e resíduo, com detecção e tratamento de falhas (derivadada nula, divisão por zero, ausência de mudança de sinal e esgotamento de iterações);
3. Aplicar os métodos a quatro funções distintas com diferentes características matemáticas ($f_1, f_2, f_3, f_4$), organizando os resultados em tabela padronizada;
4. Produzir animações gráficas (exportadas como arquivos GIF) ilustrando a dinâmica geométrica de convergência de cada método;
5. Comparar taxas de convergência, custo computacional e sensibilidade às condições iniciais.



## 2. Fundamentação Teórica

### 2.1 O Problema dos Zeros de Funções
Dada uma função contínua $f: [a, b] \to \mathbb{R}$, busca-se $r \in [a, b]$ tal que:
$$ f(r) = 0 $$

Pelo **Teorema do Valor Intermediário (TVI / Teorema de Bolzano)**: se $f$ é contínua em $[a, b]$ e $f(a) \cdot f(b) < 0$, então existe pelo menos um valor $r \in (a, b)$ tal que $f(r) = 0$.

---

### 2.2 Método da Bisseção
O método da bisseção baseia-se na repetida divisão binária de um intervalo $[a, b]$ que contém uma raiz. A cada iteração $k$:
1. Calcula-se o ponto médio:
   $$ m_k = \frac{a_k + b_k}{2} = a_k + \frac{b_k - a_k}{2} $$
2. Avalia-se $f(m_k)$. Se $f(a_k) \cdot f(m_k) < 0$, a raiz está no subintervalo esquerdo $[a_k, m_k]$; caso contrário, está em $[m_k, b_k]$.
3. O tamanho do intervalo reduz-se pela metade a cada passo:
   $$ |b_k - a_k| = \frac{|b_0 - a_0|}{2^k} $$

- **Ordem de Convergência:** Linear ($p = 1$, com fator assintótico $C = 1/2$).
- **Número Teórico de Iterações:** Para garantir um erro menor que uma tolerância $\text{tol}$, tem-se:
  $$ \frac{b_0 - a_0}{2^k} \le \text{tol} \implies k \ge \frac{\ln(b_0 - a_0) - \ln(\text{tol})}{\ln 2} $$
  Para $b_0 - a_0 = 1$ e $\text{tol} = 10^{-6}$:
  $$ k \ge \frac{\ln(10^6)}{\ln 2} \approx \frac{13.8155}{0.6931} \approx 19.93 \implies k = 20\text{ iterações}. $$
- **Vantagens:** Convergência global incondicional garantida se $f(a)f(b) < 0$.
- **Desvantagens:** Convergência relativamente lenta.

---

### 2.3 Método de Newton-Raphson
O método de Newton aproxima $f(x)$ localmente pela reta tangente em $(x_k, f(x_k))$ por meio do polinômio de Taylor de grau 1:
$$ f(x) \approx f(x_k) + f'(x_k)(x - x_k) = 0 \implies x_{k+1} = x_k - \frac{f(x_k)}{f'(x_k)} $$

- **Ordem de Convergência:** Quadrática ($p = 2$) para raízes simples ($f'(r) \neq 0$). O número de casas decimais corretas aproximadamente dobra a cada iteração próximo da raiz:
  $$ e_{k+1} \approx \frac{f''(r)}{2 f'(r)} e_k^2 $$
- **Vantagens:** Convergência ultrarrápida nas proximidades da raiz.
- **Desvantagens:** Requer o cálculo analítico da derivada $f'(x)$; pode falhar se $f'(x_k) \approx 0$ (reta tangente paralela ao eixo $x$); forte sensibilidade ao chute inicial $x_0$ (risco de oscilações ou divergência).

---

### 2.4 Método da Secante
O método da secante contorna a necessidade de avaliar a derivada exata $f'(x)$ aproximando-a pelo quociente de diferenças finitas entre as duas últimas iterações:
$$ f'(x_k) \approx \frac{f(x_k) - f(x_{k-1})}{x_k - x_{k-1}} $$
Substituindo na fórmula de Newton:
$$ x_{k+1} = x_k - f(x_k) \frac{x_k - x_{k-1}}{f(x_k) - f(x_{k-1})} $$

- **Ordem de Convergência:** Superlinear, dada pelo número de ouro $\phi = \frac{1 + \sqrt{5}}{2} \approx 1.618$.
- **Vantagens:** Não exige o cálculo da derivada $f'(x)$ e realiza apenas **uma avaliação de função por iteração** (reaproveitando $f(x_{k-1})$ do passo anterior), sendo computacionalmente muito eficiente.
- **Desvantagens:** Requer duas aproximações iniciais $(x_0, x_1)$; suscetível a cancelamento catastrófico / divisão por zero se $f(x_k) \approx f(x_{k-1})$.

---

### 2.5 Critérios de Parada Adotados
Neste trabalho, adotamos um critério de parada robusto que monitora simultaneamente:
1. **O Resíduo:** $|f(x_k)| < \text{tol}$, assegurando que a imagem da aproximação está suficientemente próxima de zero.
2. **A Variação entre Aproximações Sucessivas:** $|x_{k+1} - x_k| < \text{tol}$ (ou o semi-intervalo $\frac{b-a}{2} < \text{tol}$ na bisseção), assegurando que o algoritmo estabilizou espacialmente.

Essa verificação combinada previne tanto a estagnação prematura em funções muito inclinadas quanto oscilações espúrias em funções quase planas na vizinhança da raiz.



## 3. Parte 1 – Implementação dos Métodos em Python

As funções a seguir foram implementadas de maneira independente, comentada e parametrizada conforme solicitado. Cada uma retorna:
- A aproximação final da raiz;
- O número de iterações realizadas;
- A lista completa com as aproximações de cada iteração;
- Uma indicação clara de convergência ou de falha.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from IPython.display import display, HTML, Image as IPyImage

# Configurações globais de exibição
pd.set_option('display.float_format', lambda x: '%.8g' % x)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 10
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.4
plt.rcParams['grid.linestyle'] = '--'

# ==============================================================================
# 1. MÉTODO DA BISSEÇÃO
# ==============================================================================
def bissecao(f, a, b, tol=1e-6, max_iter=100):
    '''
    Determina a raiz de f(x) = 0 no intervalo [a, b] pelo Método da Bisseção.
    
    Retorna:
        raiz (float ou None): Aproximação final da raiz.
        iteracoes (int): Número de iterações executadas.
        aproximacoes (list): Lista com os pontos médios calculados em cada passo.
        status (str): Mensagem de convergência ou motivo de falha.
    '''
    fa, fb = f(a), f(b)
    
    # Detecção de falha: ausência de mudança de sinal (TVI/Bolzano não satisfeito)
    if fa * fb > 0:
        return None, 0, [], "Falha: f(a)*f(b) > 0 (sem garantia de raiz pelo Teorema de Bolzano)"
    
    # Casos triviais de contorno
    if abs(fa) < 1e-15:
        return a, 0, [a], "Convergência: a extremidade 'a' é raiz exata"
    if abs(fb) < 1e-15:
        return b, 0, [b], "Convergência: a extremidade 'b' é raiz exata"
        
    aproximacoes = []
    
    for k in range(1, max_iter + 1):
        # Ponto médio (usando formulação numericamente estável que evita overflow em somas grandes)
        m = a + (b - a) / 2.0
        fm = f(m)
        aproximacoes.append(m)
        
        # Critério de parada composto: resíduo ou semi-intervalo
        if abs(fm) < tol or (b - a) / 2.0 < tol:
            return m, k, aproximacoes, "Convergência atingida"
            
        # Atualização do intervalo
        if fa * fm < 0:
            b, fb = m, fm
        else:
            a, fa = m, fm
            
    return m, max_iter, aproximacoes, "Falha: atingiu o número máximo de iterações"


# ==============================================================================
# 2. MÉTODO DE NEWTON-RAPHSON
# ==============================================================================
def newton(f, df, x0, tol=1e-6, max_iter=100, deriv_tol=1e-14):
    '''
    Determina a raiz de f(x) = 0 a partir de x0 pelo Método de Newton-Raphson.
    
    Retorna:
        raiz (float ou None): Aproximação final da raiz.
        iteracoes (int): Número de iterações executadas.
        aproximacoes (list): Lista com as aproximações em cada passo.
        status (str): Mensagem de convergência ou motivo de falha.
    '''
    x = float(x0)
    aproximacoes = [x]
    
    fx = f(x)
    if abs(fx) < tol:
        return x, 0, aproximacoes, "Convergência: x0 já satisfaz a tolerância"
        
    for k in range(1, max_iter + 1):
        dfx = df(x)
        
        # Detecção de falha: derivada nula ou excessivamente pequena
        if abs(dfx) <= deriv_tol:
            return x, k - 1, aproximacoes, f"Falha: derivada próxima de zero (|f'({x:.4f})| <= {deriv_tol})"
            
        x_novo = x - fx / dfx
        aproximacoes.append(x_novo)
        fx_novo = f(x_novo)
        
        # Critério de parada: passo pequeno ou resíduo pequeno
        if abs(x_novo - x) < tol or abs(fx_novo) < tol:
            return x_novo, k, aproximacoes, "Convergência atingida"
            
        x, fx = x_novo, fx_novo
        
    return x, max_iter, aproximacoes, "Falha: atingiu o número máximo de iterações"


# ==============================================================================
# 3. MÉTODO DA SECANTE
# ==============================================================================
def secante(f, x0, x1, tol=1e-6, max_iter=100, denom_tol=1e-14):
    '''
    Determina a raiz de f(x) = 0 a partir de (x0, x1) pelo Método da Secante.
    
    Retorna:
        raiz (float ou None): Aproximação final da raiz.
        iteracoes (int): Número de iterações executadas.
        aproximacoes (list): Lista com as aproximações em cada passo.
        status (str): Mensagem de convergência ou motivo de falha.
    '''
    xa, xb = float(x0), float(x1)
    fa, fb = f(xa), f(xb)
    aproximacoes = [xa, xb]
    
    if abs(fb) < tol:
        return xb, 0, aproximacoes, "Convergência: x1 já satisfaz a tolerância"
        
    for k in range(1, max_iter + 1):
        denominador = fb - fa
        
        # Detecção de falha: denominador nulo ou muito próximo de zero
        if abs(denominador) <= denom_tol:
            return xb, k - 1, aproximacoes, f"Falha: denominador muito próximo de zero (|f(x_k) - f(x_{{k-1}})| <= {denom_tol})"
            
        x_novo = xb - fb * (xb - xa) / denominador
        aproximacoes.append(x_novo)
        fx_novo = f(x_novo)
        
        # Critério de parada: passo pequeno ou resíduo pequeno
        if abs(x_novo - xb) < tol or abs(fx_novo) < tol:
            return x_novo, k, aproximacoes, "Convergência atingida"
            
        xa, fa = xb, fb
        xb, fb = x_novo, fx_novo
        
    return xb, max_iter, aproximacoes, "Falha: atingiu o número máximo de iterações"

print("Funções 'bissecao', 'newton' e 'secante' implementadas e validadas!")



---
## 4. Parte 2 – Aplicação e Análise

Aplicamos os três métodos às quatro funções propostas no enunciado:
1. $f_1(x) = x^2 - 2, \quad f_1'(x) = 2x$
2. $f_2(x) = x^3 - x - 2, \quad f_2'(x) = 3x^2 - 1$
3. $f_3(x) = e^{-x} - x, \quad f_3'(x) = -e^{-x} - 1$
4. $f_4(x) = x^3 - 2x + 2, \quad f_4'(x) = 3x^2 - 2$

Com as condições iniciais sugeridas na tabela do PDF:
- Bisseção: Intervalos $[1, 2]$, $[1, 2]$, $[0, 1]$ e $[-2, -1]$.
- Newton: Chutes $x_0 = 1.5, 1.5, 0.5$ e $-2.0$.
- Secante: Pares $(1, 2), (1, 2), (0, 1)$ e $(-2, -1)$.



In [ ]:
# Definição analítica das funções e derivadas
funcoes_teste = [
    {
        "nome": "f1(x) = x² - 2",
        "latex": "$f_1(x) = x^2 - 2$",
        "f": lambda x: x**2 - 2.0,
        "df": lambda x: 2.0 * x,
        "bis_int": (1.0, 2.0),
        "newton_x0": 1.5,
        "sec_pts": (1.0, 2.0)
    },
    {
        "nome": "f2(x) = x³ - x - 2",
        "latex": "$f_2(x) = x^3 - x - 2$",
        "f": lambda x: x**3 - x - 2.0,
        "df": lambda x: 3.0 * (x**2) - 1.0,
        "bis_int": (1.0, 2.0),
        "newton_x0": 1.5,
        "sec_pts": (1.0, 2.0)
    },
    {
        "nome": "f3(x) = e^(-x) - x",
        "latex": "$f_3(x) = e^{-x} - x$",
        "f": lambda x: np.exp(-x) - x,
        "df": lambda x: -np.exp(-x) - 1.0,
        "bis_int": (0.0, 1.0),
        "newton_x0": 0.5,
        "sec_pts": (0.0, 1.0)
    },
    {
        "nome": "f4(x) = x³ - 2x + 2",
        "latex": "$f_4(x) = x^3 - 2x + 2$",
        "f": lambda x: x**3 - 2.0 * x + 2.0,
        "df": lambda x: 3.0 * (x**2) - 2.0,
        "bis_int": (-2.0, -1.0),
        "newton_x0": -2.0,
        "sec_pts": (-2.0, -1.0)
    }
]

# Execução sistemática dos testes
tabela_resultados = []

for item in funcoes_teste:
    f, df = item["f"], item["df"]
    nome = item["nome"]
    
    # 1. Bisseção
    a, b = item["bis_int"]
    r_b, it_b, hist_b, st_b = bissecao(f, a, b, tol=1e-6, max_iter=100)
    tabela_resultados.append({
        "Função": nome,
        "Método": "Bisseção",
        "Valores iniciais": f"[{a:.1f}, {b:.1f}]",
        "Raiz aproximada": r_b,
        "|f(x)| final": abs(f(r_b)) if r_b is not None else np.nan,
        "Iterações/resultado": f"{it_b} iterações ({st_b})"
    })
    
    # 2. Newton-Raphson
    x0 = item["newton_x0"]
    r_n, it_n, hist_n, st_n = newton(f, df, x0, tol=1e-6, max_iter=100)
    tabela_resultados.append({
        "Função": nome,
        "Método": "Newton",
        "Valores iniciais": f"x0 = {x0:.1f}",
        "Raiz aproximada": r_n,
        "|f(x)| final": abs(f(r_n)) if r_n is not None else np.nan,
        "Iterações/resultado": f"{it_n} iterações ({st_n})"
    })
    
    # 3. Secante
    x0_s, x1_s = item["sec_pts"]
    r_s, it_s, hist_s, st_s = secante(f, x0_s, x1_s, tol=1e-6, max_iter=100)
    tabela_resultados.append({
        "Função": nome,
        "Método": "Secante",
        "Valores iniciais": f"({x0_s:.1f}, {x1_s:.1f})",
        "Raiz aproximada": r_s,
        "|f(x)| final": abs(f(r_s)) if r_s is not None else np.nan,
        "Iterações/resultado": f"{it_s} iterações ({st_s})"
    })

df_resultados = pd.DataFrame(tabela_resultados)

print("="*95)
print("TABELA CONSOLIDADA DE RESULTADOS (PARTE 2)")
print("="*95)
display(df_resultados)



In [ ]:
# Gráficos de Convergência do Erro/Resíduo ao longo das iterações
fig, axs = plt.subplots(2, 2, figsize=(15, 10))

for idx, item in enumerate(funcoes_teste):
    ax = axs[idx // 2, idx % 2]
    f, df = item["f"], item["df"]
    
    _, _, hist_b, _ = bissecao(f, item["bis_int"][0], item["bis_int"][1], tol=1e-6)
    _, _, hist_n, _ = newton(f, df, item["newton_x0"], tol=1e-6)
    _, _, hist_s, _ = secante(f, item["sec_pts"][0], item["sec_pts"][1], tol=1e-6)
    
    res_b = [abs(f(x)) for x in hist_b]
    res_n = [abs(f(x)) for x in hist_n]
    res_s = [abs(f(x)) for x in hist_s]
    
    ax.semilogy(range(1, len(res_b) + 1), res_b, 'o-', label=f'Bisseção ({len(hist_b)} it.)', color='#d62728', lw=1.8)
    ax.semilogy(range(0, len(res_n)), res_n, 's--', label=f'Newton ({len(hist_n)-1} it.)', color='#1f77b4', lw=2)
    ax.semilogy(range(1, len(res_s)), res_s[1:], '^-.', label=f'Secante ({len(hist_s)-2} it.)', color='#2ca02c', lw=1.8)
    
    ax.axhline(1e-6, color='black', ls=':', label='Tolerância (1e-6)')
    ax.set_title(item["nome"], weight='bold')
    ax.set_xlabel("Iterações")
    ax.set_ylabel("Resíduo |f(x)| (escala log)")
    ax.legend()

plt.suptitle("Comparativo da Taxa de Convergência do Resíduo |f(x_k)|", fontsize=15, weight='bold')
plt.tight_layout()
plt.show()



### Discussão Comparativa dos Resultados

1. **Velocidade de Convergência e Número de Iterações:**
   - **Bisseção:** Conforme previsto teoricamente, para um intervalo inicial de amplitude 1 ($|b_0 - a_0| = 1$), a bisseção requer rigorosamente **20 iterações** para todas as funções:
     $$ \frac{1}{2^{20}} = \frac{1}{1048576} \approx 9.537 \times 10^{-7} < 10^{-6} $$
     A convergência é estritamente previsível e puramente geométrica, independente da curvatura da função.
   - **Newton-Raphson:** Atingiu a tolerância em apenas **2 a 4 iterações**. Sua taxa de convergência quadrática faz com que os resíduos atinjam ordens de $10^{-7}$ até $10^{-13}$ instantaneamente nas proximidades da raiz.
   - **Secante:** Convergiu entre **4 e 6 iterações**, demonstrando sua ordem superlinear ($p \approx 1.618$).

2. **Custo Computacional Efetivo:**
   - Embora o método de Newton use menos iterações que o método da Secante, ele exige **duas avaliações analíticas por iteração** ($f(x_k)$ e $f'(x_k)$).
   - O método da Secante utiliza apenas **uma avaliação de função por iteração** (pois armazena $f(x_{k-1})$ do passo anterior), sem necessitar da derivada analítica.
   - Em problemas reais onde avaliar $f'(x)$ é custoso ou impossível analiticamente, o método da Secante costuma ser mais eficiente em tempo de CPU do que o método de Newton.



---
## 5. Parte 3 – Animações Gráficas das Iterações

Conforme exigido pelo PDF, criamos animações para três combinações representativas de método e função:
1. **Animação 1 (Bisseção):** Aplicada a $f_2(x) = x^3 - x - 2$ no intervalo $[1, 2]$. Mostra o intervalo que contém a raiz e sua redução progressiva com o ponto médio $m_k$.
2. **Animação 2 (Newton-Raphson):** Aplicada a $f_1(x) = x^2 - 2$ iniciando em $x_0 = 2.0$. Mostra as retas tangentes à curva cruzando o eixo das abscissas.
3. **Animação 3 (Secante):** Aplicada a $f_4(x) = x^3 - 2x + 2$ iniciando no par $(-2, -1)$. Mostra as retas secantes passando pelos dois pontos a cada iteração.

Todas as animações apresentam:
- O gráfico de $f(x)$ e o eixo horizontal $y = 0$;
- Os pontos correspondentes às aproximações geradas;
- Os segmentos, intervalos ou retas empregados;
- O número da iteração atual;
- O valor atual da aproximação e o resíduo $|f(x_k)|$;
- Exportação direta para arquivos `.gif` reproduzíveis.



In [ ]:
# ==============================================================================
# 1. GERAÇÃO DO GIF: MÉTODO DA BISSEÇÃO
# ==============================================================================
def criar_gif_bissecao(nome_arquivo="animacao_bissecao.gif"):
    f = lambda x: x**3 - x - 2.0
    a, b = 1.0, 2.0
    frames = []
    
    xs = np.linspace(0.8, 2.2, 400)
    ys = f(xs)
    fa, fb = f(a), f(b)
    
    for k in range(1, 9):
        m = a + (b - a) / 2.0
        fm = f(m)
        
        fig, ax = plt.subplots(figsize=(9, 5.5), dpi=100)
        ax.plot(xs, ys, "b-", lw=2.2, label="$f_2(x) = x^3 - x - 2$")
        ax.axhline(0, color="black", lw=1.2)
        
        # Região do intervalo [a, b]
        ax.axvspan(a, b, color="#ffa600", alpha=0.25, label=f"Intervalo $[a_{k}, b_{k}]$")
        ax.plot([a, b], [0, 0], "o", color="#d45087", markersize=7, label="Extremidades $a, b$")
        ax.plot([m], [0], "ro", markersize=8, label=f"Ponto médio $m_{k}$ ({m:.4f})")
        ax.plot([m], [fm], "rs", markersize=7)
        ax.plot([m, m], [0, fm], "r--", lw=1.4)
        
        # Painel HUD com informações do processo
        hud = (
            f"MÉTODO DA BISSEÇÃO\n"
            f"Iteração: {k}\n"
            f"Intervalo [a, b]: [{a:.5f}, {b:.5f}]\n"
            f"Amplitude (b - a): {b - a:.5f}\n"
            f"Ponto médio m: {m:.5f}\n"
            f"Resíduo |f(m)|: {abs(fm):.4e}"
        )
        ax.text(0.03, 0.95, hud, transform=ax.transAxes, verticalalignment="top",
                bbox=dict(boxstyle="round,pad=0.6", facecolor="#f8f9fa", edgecolor="#6c757d", alpha=0.95),
                fontsize=10, family="monospace")
        
        ax.set_xlim(0.8, 2.2)
        ax.set_ylim(-3, 5)
        ax.set_xlabel("x", weight="bold")
        ax.set_ylabel("f(x)", weight="bold")
        ax.set_title(f"Método da Bisseção - Iteração {k} (Redução de Intervalo)", weight="bold", fontsize=12)
        ax.legend(loc="lower right", frameon=True)
        
        plt.tight_layout()
        fig.canvas.draw()
        rgba = np.array(fig.canvas.buffer_rgba())
        frames.append(Image.fromarray(rgba))
        plt.close(fig)
        
        if fa * fm < 0:
            b, fb = m, fm
        else:
            a, fa = m, fm
            
    frames[0].save(nome_arquivo, save_all=True, append_images=frames[1:], duration=800, loop=0)
    print(f"-> Arquivo '{nome_arquivo}' gerado com sucesso!")

criar_gif_bissecao()
display(IPyImage("animacao_bissecao.gif"))



In [ ]:
# ==============================================================================
# 2. GERAÇÃO DO GIF: MÉTODO DE NEWTON-RAPHSON
# ==============================================================================
def criar_gif_newton(nome_arquivo="animacao_newton.gif"):
    f = lambda x: x**2 - 2.0
    df = lambda x: 2.0 * x
    x = 2.0
    frames = []
    
    xs = np.linspace(0.8, 2.4, 400)
    ys = f(xs)
    
    for k in range(1, 6):
        fx = f(x)
        dfx = df(x)
        x_prox = x - fx / dfx
        
        fig, ax = plt.subplots(figsize=(9, 5.5), dpi=100)
        ax.plot(xs, ys, "b-", lw=2.2, label="$f_1(x) = x^2 - 2$")
        ax.axhline(0, color="black", lw=1.2)
        
        # Ponto atual na curva
        ax.plot([x], [fx], "ro", markersize=8, label=f"Ponto $(x_{{{k-1}}}, f(x_{{{k-1}}}))$")
        ax.plot([x, x], [0, fx], "r--", lw=1.4)
        ax.plot([x], [0], "rs", markersize=6)
        
        # Reta tangente à curva
        tg_xs = np.linspace(max(0.8, x_prox - 0.3), min(2.4, x + 0.3), 100)
        tg_ys = fx + dfx * (tg_xs - x)
        ax.plot(tg_xs, tg_ys, "g--", lw=2.0, label=f"Reta Tangente em $x_{{{k-1}}}$")
        
        # Nova aproximação no cruzamento com y = 0
        ax.plot([x_prox], [0], "g*", markersize=13, label=f"Nova raiz $x_{{{k}}} = {x_prox:.5f}$")
        
        hud = (
            f"MÉTODO DE NEWTON-RAPHSON\n"
            f"Iteração: {k}\n"
            f"Aproximação x_{{{k-1}}}: {x:.6f}\n"
            f"Derivada f'(x_{{{k-1}}}): {dfx:.4f}\n"
            f"Próximo x_{{{k}}}: {x_prox:.6f}\n"
            f"Resíduo |f(x_{{{k-1}}})|: {abs(fx):.4e}"
        )
        ax.text(0.03, 0.95, hud, transform=ax.transAxes, verticalalignment="top",
                bbox=dict(boxstyle="round,pad=0.6", facecolor="#f8f9fa", edgecolor="#6c757d", alpha=0.95),
                fontsize=10, family="monospace")
        
        ax.set_xlim(0.8, 2.4)
        ax.set_ylim(-2, 4)
        ax.set_xlabel("x", weight="bold")
        ax.set_ylabel("f(x)", weight="bold")
        ax.set_title(f"Método de Newton-Raphson - Iteração {k} (Retas Tangentes)", weight="bold", fontsize=12)
        ax.legend(loc="lower right", frameon=True)
        
        plt.tight_layout()
        fig.canvas.draw()
        rgba = np.array(fig.canvas.buffer_rgba())
        frames.append(Image.fromarray(rgba))
        plt.close(fig)
        
        x = x_prox
        
    frames[0].save(nome_arquivo, save_all=True, append_images=frames[1:], duration=900, loop=0)
    print(f"-> Arquivo '{nome_arquivo}' gerado com sucesso!")

criar_gif_newton()
display(IPyImage("animacao_newton.gif"))



In [ ]:
# ==============================================================================
# 3. GERAÇÃO DO GIF: MÉTODO DA SECANTE
# ==============================================================================
def criar_gif_secante(nome_arquivo="animacao_secante.gif"):
    f = lambda x: x**3 - 2.0 * x + 2.0
    x0, x1 = -2.0, -1.0
    frames = []
    
    xs = np.linspace(-2.5, -0.6, 400)
    ys = f(xs)
    
    xa, xb = x0, x1
    fa, fb = f(xa), f(xb)
    
    for k in range(1, 7):
        slope = (fb - fa) / (xb - xa)
        x_prox = xb - fb / slope
        
        fig, ax = plt.subplots(figsize=(9, 5.5), dpi=100)
        ax.plot(xs, ys, "b-", lw=2.2, label="$f_4(x) = x^3 - 2x + 2$")
        ax.axhline(0, color="black", lw=1.2)
        
        # Pontos base da secante
        ax.plot([xa, xb], [fa, fb], "ro", markersize=7, label="Pontos da Secante")
        ax.plot([xa, xa], [0, fa], "r--", lw=1.2)
        ax.plot([xb, xb], [0, fb], "r--", lw=1.2)
        
        # Reta secante estendida
        sec_xs = np.linspace(-2.4, -0.7, 100)
        sec_ys = fb + slope * (sec_xs - xb)
        ax.plot(sec_xs, sec_ys, "m--", lw=2.0, label="Reta Secante")
        
        # Nova aproximação no eixo x
        ax.plot([x_prox], [0], "g*", markersize=13, label=f"Nova raiz $x_{{{k+1}}} = {x_prox:.5f}$")
        
        hud = (
            f"MÉTODO DA SECANTE\n"
            f"Iteração: {k}\n"
            f"Ponto x_{{{k-1}}}: {xa:.5f} (f={fa:.3e})\n"
            f"Ponto x_{{{k}}}:   {xb:.5f} (f={fb:.3e})\n"
            f"Próximo x_{{{k+1}}}: {x_prox:.5f}\n"
            f"Resíduo |f(x_{{{k}}})|: {abs(fb):.4e}"
        )
        ax.text(0.03, 0.95, hud, transform=ax.transAxes, verticalalignment="top",
                bbox=dict(boxstyle="round,pad=0.6", facecolor="#f8f9fa", edgecolor="#6c757d", alpha=0.95),
                fontsize=10, family="monospace")
        
        ax.set_xlim(-2.5, -0.6)
        ax.set_ylim(-5, 6)
        ax.set_xlabel("x", weight="bold")
        ax.set_ylabel("f(x)", weight="bold")
        ax.set_title(f"Método da Secante - Iteração {k} (Retas Secantes)", weight="bold", fontsize=12)
        ax.legend(loc="lower right", frameon=True)
        
        plt.tight_layout()
        fig.canvas.draw()
        rgba = np.array(fig.canvas.buffer_rgba())
        frames.append(Image.fromarray(rgba))
        plt.close(fig)
        
        xa, fa = xb, fb
        xb, fb = x_prox, f(x_prox)
        
    frames[0].save(nome_arquivo, save_all=True, append_images=frames[1:], duration=900, loop=0)
    print(f"-> Arquivo '{nome_arquivo}' gerado com sucesso!")

criar_gif_secante()
display(IPyImage("animacao_secante.gif"))



---
## 6. Análise de Robustez e Detecção de Falhas

O enunciado destaca que o algoritmo deve ser capaz de detectar situações impeditivas de continuidade. Implementamos testes específicos para comprovar a eficácia dessa detecção:
1. **Bisseção sem mudança de sinal:** $f(a)f(b) > 0$.
2. **Newton com derivada nula / horizontal:** $f'(x_k) \approx 0$.
3. **Secante com divisão por zero:** $f(x_k) - f(x_{k-1}) \approx 0$.



In [ ]:
print("="*80)
print("TESTES DE DETECÇÃO DE FALHAS E TRATAMENTO DE ERROS")
print("="*80)

# 1. Bisseção com intervalo inválido
f_exemplo = lambda x: x**2 + 1
r, it, _, st = bissecao(f_exemplo, 1.0, 2.0)
print(f"[TESTE BISSEÇÃO] f(x) = x² + 1 em [1, 2] -> Status: {st}")

# 2. Newton com derivada nula (ponto de mínimo/máximo)
# Para f4(x) = x³ - 2x + 2, f'(x) = 3x² - 2 = 0 quando x = sqrt(2/3) ≈ 0.81649658
x_critico = np.sqrt(2.0 / 3.0)
f_crit = lambda x: x**3 - 2.0*x + 2.0
df_crit = lambda x: 3.0*x**2 - 2.0
r, it, _, st = newton(f_crit, df_crit, x_critico)
print(f"[TESTE NEWTON] Ponto crítico x0 = {x_critico:.6f} -> Status: {st}")

# 3. Secante com denominador nulo (pontos simétricos com f(x0) = f(x1))
f_par = lambda x: x**2 - 4.0
r, it, _, st = secante(f_par, -2.5, 2.5)
print(f"[TESTE SECANTE] Pontos simétricos x0 = -2.5, x1 = 2.5 -> Status: {st}")



---
## 7. Síntese dos Entregáveis e Orientações

Conforme a seção **Entregáveis** do documento PDF da Atividade Prática III:
> *"Cada estudante ou grupo deverá entregar um dashboard ilustrativo de resultado no https://www.mathcha.io/ contendo:*
> 1. *Link para o notebook do Google Colab com as implementações e os testes;*
> 2. *Os arquivos GIF das animações;*
> 3. *A tabela de resultados."*

### Elementos Prontos para Exportação:
1. **Link do Google Colab:**
   O notebook foi totalmente estruturado para execução imediata no Google Colab, bastando clicar no badge interativo no topo ou utilizar a URL:
   `https://colab.research.google.com/github/brenoprato/calculo-numerico/blob/main/atividades/atv_3/resolucao.ipynb`
2. **Arquivos GIF Gerados:**
   - `animacao_bissecao.gif` (Método da Bisseção em $f_2$)
   - `animacao_newton.gif` (Método de Newton em $f_1$)
   - `animacao_secante.gif` (Método da Secante em $f_4$)
3. **Tabela de Resultados:**
   Estruturada na Parte 2 em conformidade com as colunas requeridas:
   `Função | Método | Valores iniciais | Raiz aproximada | |f(x)| final | Iterações/resultado`

---

## 8. Conclusão Final

1. **Eficiência vs. Robustez:**
   - O **Método da Bisseção** é o mais robusto e previsível, com garantia absoluta de convergência sob o Teorema de Bolzano. Seu número de iterações pode ser estritamente calculado a priori ($k=20$ para $\text{tol} = 10^{-6}$), sendo ideal para localizar intervalos e fornecer chutes iniciais confiáveis.
   - O **Método de Newton-Raphson** é o mais rápido (convergência quadrática, 2 a 4 iterações), porém é vulnerável a derivadas nulas e requer o conhecimento analítico de $f'(x)$.
   - O **Método da Secante** oferece o melhor equilíbrio prático em muitas aplicações: velocidade superlinear ($p \approx 1.618$) sem a necessidade de calcular derivadas, exigindo apenas uma avaliação de função por passo.

2. **Modularidade e Extensibilidade:**
   O código foi desenvolvido de forma modular e parametrizável, permitindo que novos testes com quaisquer funções contínuas, tolerâncias e condições iniciais sejam executados de forma direta e sem alterações na lógica interna dos métodos.

